# Stable Diffusion

**Goal**: Using Stable diffusion to create images from text descriptions. Starts by working with each of the components of Stable Diffusion individually. Finishes by using Stable Diffusion pipeline that joins all of the components together, providing a much more convenient interface.

**Objectives**

- Use the CLIP model to create text embeddings.
- Decode latent vectors into an image with a VAE.
- Apply a denoising diffusion model to randomly-generated noise.
- Use a scheduler to perform denoising over multiple steps.
- Generate images matching a text description with a Stable Diffusion pipeline.

In [11]:
import sys

import diffusers
import matplotlib.pyplot as plt 
import torch
import transformers
from IPython.display import display
from PIL import Image
from torchinfo import summary
from tqdm.notebook import tqdm

In [12]:
print("Platform:", sys.platform)
print("Python version:", sys.version)
print("---")
print("diffusers version:", diffusers.__version__)
print("transformers version:", transformers.__version__)
print("torch version:", torch.__version__)
print("PIL version:", Image.__version__)

Platform: win32
Python version: 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)]
---
diffusers version: 0.33.1
transformers version: 4.51.3
torch version: 2.2.2+cpu
PIL version: 10.2.0


In [13]:
if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
else:
    device = "cpu"
    dtype = torch.float32

print(f"Using {device} device with {dtype} data type.")

Using cpu device with torch.float32 data type.


**Creating Text Embeddings**

Stable Diffusion attempts to produce an image aligned to a text description. We start by working on that text description, by converting human-readable text into an embedding. This means that the text will be represented by a bunch of numbers, which the model will be able to ingest. 

Stable Diffusion uses text embeddings from the Constrastive Language-Image Pre-Training (CLIP) model. We need two pieces from this model; first is the tokenizer that splits up a string into tokens, a word, part of a word, or single character.

In [15]:
tokenizer = transformers.CLIPTokenizer.from_pretrained(
    "CompVis/stable-diffusion-v1-4", subfolder="tokenizer",
    torch_dtype=dtype
)
print(tokenizer)

CLIPTokenizer(name_or_path='CompVis/stable-diffusion-v1-4', vocab_size=49408, model_max_length=77, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	49406: AddedToken("<|startoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	49407: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)


In [16]:
type(tokenizer)

transformers.models.clip.tokenization_clip.CLIPTokenizer

Example usage:

In [17]:
text = "Hello, world!"
result = tokenizer(text)

print(type(result))
print(result)

<class 'transformers.tokenization_utils_base.BatchEncoding'>
{'input_ids': [49406, 3306, 267, 1002, 256, 49407], 'attention_mask': [1, 1, 1, 1, 1, 1]}


In [18]:
result["input_ids"]

[49406, 3306, 267, 1002, 256, 49407]

In [19]:
result.input_ids

[49406, 3306, 267, 1002, 256, 49407]

Decode Tokens:

In [20]:
for token in result.input_ids:
    print(tokenizer.decode(token))

<|startoftext|>
hello
,
world
!
<|endoftext|>


**Image generation prompt**